In [1]:
import os
import json
import pandas as pd
import torch
import torch.nn as nn
from torch.backends import cudnn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from torchvision.transforms import v2, InterpolationMode
from PIL import Image

torch.manual_seed(3407)
torch.cuda.manual_seed_all(3407)
cudnn.deterministic = False
cudnn.benchmark = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

train_csv_path = "/kaggle/input/cassava-leaf-disease-classification/train.csv"
test_dir = "/kaggle/input/cassava-leaf-disease-classification/test_images/"
sample_sub_path = (
    "/kaggle/input/cassava-leaf-disease-classification/sample_submission.csv"
)




Using device: cuda


In [2]:
class CassavaDataset(Dataset):
    def __init__(self, csv_path=None, img_dir=None, transform=None, is_test=False):
        self.is_test = is_test
        self.transform = transform
        self.img_dir = img_dir
        if not is_test:
            self.df = pd.read_csv(csv_path)
            self.images = self.df["image_id"].tolist()
            self.labels = self.df["label"].tolist()
        else:
            # Keep deterministic order but exclude sub‑directories or non‑image files
            self.images = sorted(
                [
                    f
                    for f in os.listdir(img_dir)
                    if os.path.isfile(os.path.join(img_dir, f))
                    and f.lower().endswith((".jpg", ".jpeg", ".png"))
                ]
            )

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.img_dir, img_name)
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        if self.is_test:
            return img, img_name
        else:
            label = self.labels[idx]
            return img, label




In [3]:
train_transforms = v2.Compose(
    [
        v2.RandomResizedCrop(384, scale=(0.8, 1.0)),
        v2.RandomHorizontalFlip(),
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

test_transforms = v2.Compose(
    [
        v2.Resize((384, 384), interpolation=InterpolationMode.BICUBIC),
        v2.CenterCrop(384),
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)




In [4]:
batch_size = 64
num_workers = 4
num_classes = 5
epochs = 5
learning_rate = 3e-4

train_dataset = CassavaDataset(
    csv_path=train_csv_path,
    img_dir="/kaggle/input/cassava-leaf-disease-classification/train_images/",
    transform=train_transforms,
    is_test=False,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True,
)




In [5]:
model = models.resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)




/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 111MB/s]


In [6]:
model.train()
for epoch in range(epochs):
    epoch_loss = 0.0
    correct = 0
    total = 0
    for imgs, labels in train_loader:
        imgs = imgs.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * imgs.size(0)

        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_acc = correct / total
    epoch_loss /= total
    print(f"Epoch [{epoch+1}/{epochs}] - Loss: {epoch_loss:.4f} - Acc: {epoch_acc:.4f}")




Epoch [1/5] - Loss: 0.5362 - Acc: 0.8106
Epoch [2/5] - Loss: 0.4081 - Acc: 0.8581
Epoch [3/5] - Loss: 0.3652 - Acc: 0.8723
Epoch [4/5] - Loss: 0.3449 - Acc: 0.8786
Epoch [5/5] - Loss: 0.3157 - Acc: 0.8914


In [7]:
model.eval()
test_dataset = CassavaDataset(
    csv_path=None, img_dir=test_dir, transform=test_transforms, is_test=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True,
)

all_names = []
all_preds = []
softmax = nn.Softmax(dim=1)

with torch.no_grad():
    for imgs, filenames in test_loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        probs = softmax(outputs)
        preds = torch.argmax(probs, dim=1).cpu().tolist()
        all_names.extend(filenames)
        all_preds.extend(preds)




In [8]:
submission = pd.DataFrame({"image_id": all_names, "label": all_preds})
sample_len = pd.read_csv(sample_sub_path).shape[0]
if submission.shape[0] != sample_len:
    submission = submission.sort_values("image_id").reset_index(drop=True)
submission.to_csv("submission.csv", index=False)
print("Saved submission.csv with", submission.shape[0], "rows")

Saved submission.csv with 2676 rows
